In [154]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [155]:
import numpy as np
import torch

from torch import nn
from tokenizers import Tokenizer

from storyforge.config import load_config
from storyforge.batching import get_batch
import math

In [156]:
config = load_config("configs/default.toml")

In [157]:
dropout_probability = config["model"]["dropout"]

In [158]:
torch.manual_seed(config["seed"])
rng = np.random.default_rng(config["seed"])

In [159]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [160]:
train_tokens = np.load(
    config["encoding"]["train_path"],
    mmap_mode="r",
)

In [161]:
inputs, targets = get_batch(
    tokens=train_tokens,
    batch_size=config["training"]["batch_size"],
    context=config["model"]["max_context"],
    rng=rng,
    device=device,
)

In [162]:
print("Inputs:", inputs.shape)
print("Targets:", targets.shape)

Inputs: torch.Size([8, 256])
Targets: torch.Size([8, 256])


In [163]:
tokenizer = Tokenizer.from_file(
    config["tokenizer"]["path"]
)


In [164]:
vocab_size = tokenizer.get_vocab_size()
d_model = config["model"]["d_model"]
max_context = config["model"]["max_context"]

In [165]:
token_embedding = nn.Embedding(
    vocab_size,
    d_model,
).to(device)

position_embedding = nn.Embedding(
    max_context,
    d_model,
).to(device)

In [166]:
positions = torch.arange(
    inputs.shape[1],
    device=device,
)


In [167]:
token_vectors = token_embedding(inputs)
position_vectors = position_embedding(positions)

In [168]:
hidden = token_vectors + position_vectors

In [169]:
qkv_projection = nn.Linear(
    d_model,
    3 * d_model,
).to(device)

In [170]:
qkv = qkv_projection(hidden)

In [171]:
q, k, v = qkv.chunk(3, dim=-1)

In [172]:
print("Combined QKV:", qkv.shape)
print("Queries:", q.shape)
print("Keys:", k.shape)
print("Values:", v.shape)

Combined QKV: torch.Size([8, 256, 1152])
Queries: torch.Size([8, 256, 384])
Keys: torch.Size([8, 256, 384])
Values: torch.Size([8, 256, 384])


In [173]:
n_head = config["model"]["n_head"]
head_dim = d_model // n_head

In [174]:
assert d_model % n_head == 0

In [175]:
batch_size, sequence_length, _ = q.shape

In [176]:
q_heads = q.reshape(
    batch_size, sequence_length, n_head, head_dim
).transpose(1, 2)

k_heads = k.reshape(
    batch_size, sequence_length, n_head, head_dim
).transpose(1, 2)

v_heads = v.reshape(
    batch_size, sequence_length, n_head, head_dim
).transpose(1, 2)

In [177]:
print("Queries:", q_heads.shape)
print("Keys:", k_heads.shape)
print("Values:", v_heads.shape)

Queries: torch.Size([8, 6, 256, 64])
Keys: torch.Size([8, 6, 256, 64])
Values: torch.Size([8, 6, 256, 64])


In [178]:
scores = (
    q_heads @ k_heads.transpose(-2, -1)
) / math.sqrt(head_dim)

In [179]:
print("Attention scores:", scores.shape)

Attention scores: torch.Size([8, 6, 256, 256])


In [180]:
allowed = torch.ones(
    sequence_length,
    sequence_length,
    dtype=torch.bool,
    device=device,
).tril()

In [181]:
scores = scores.masked_fill(
    ~allowed,
    float("-inf"),
)

In [182]:
scores = scores.masked_fill(
    ~allowed,
    float("-inf"),
)

In [183]:
print(allowed[:4, :4])

tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]], device='mps:0')


In [184]:
weights = torch.softmax(
    scores.float(),
    dim=-1,
).to(v_heads.dtype)

In [185]:
print("Attention weights:", weights.shape)
print(weights[0, 0, :4, :4].detach().cpu())

Attention weights: torch.Size([8, 6, 256, 256])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.2871, 0.7129, 0.0000, 0.0000],
        [0.2808, 0.2263, 0.4929, 0.0000],
        [0.3307, 0.2233, 0.2704, 0.1756]])


In [186]:
assert torch.allclose(
    weights.sum(dim=-1),
    torch.ones_like(weights.sum(dim=-1)),
    atol=1e-5,
)

In [187]:
assert torch.all(weights.triu(diagonal=1) == 0)

In [188]:
attn_drop = nn.Dropout(dropout_probability).to(device)

dropped_weights = attn_drop(weights)
head_outputs = dropped_weights @ v_heads

In [189]:
reordered = head_outputs.transpose(1, 2)

In [190]:
print("After transpose:", reordered.shape)

After transpose: torch.Size([8, 256, 6, 64])


In [191]:
reordered = reordered.contiguous()

combined = reordered.reshape(
    batch_size,
    sequence_length,
    d_model,
)


In [192]:
print("Combined heads:", combined.shape)

Combined heads: torch.Size([8, 256, 384])


In [193]:
output_projection = nn.Linear(
    d_model,
    d_model,
).to(device)

In [194]:
out_drop = nn.Dropout(dropout_probability).to(device)

attention_output = out_drop(
    output_projection(combined)
)

print("Attention output:", attention_output.shape)

Attention output: torch.Size([8, 256, 384])


In [195]:
after_attention = hidden + attention_output

In [196]:
print("After attention residual:", after_attention.shape)

After attention residual: torch.Size([8, 256, 384])


In [197]:
norm1 = nn.LayerNorm(d_model).to(device)

In [198]:
normalized_hidden = norm1(hidden)

In [199]:
print("Shape:", normalized_hidden.shape)

Shape: torch.Size([8, 256, 384])


In [200]:
one_position = normalized_hidden[0, 0]

In [201]:
print("Mean:", one_position.mean().item())
print("Variance:", one_position.var(unbiased=False).item())

Mean: -4.967053879312289e-09
Variance: 0.9999954700469971


In [202]:
mlp_dim = d_model * config["model"]["mlp_multiplier"]

In [203]:
norm2 = nn.LayerNorm(d_model).to(device)

In [204]:
mlp = nn.Sequential(
    nn.Linear(d_model, mlp_dim),
    nn.GELU(),
    nn.Linear(mlp_dim, d_model),
    nn.Dropout(dropout_probability),
).to(device)

In [ ]:
normalized_after_attention = norm2(after_attention)
mlp_update = mlp(normalized_after_attention)
block_output = after_attention + mlp_update

In [206]:
from storyforge.attention import CausalAttention

attention = CausalAttention(config).to(device)
attention.eval()

with torch.no_grad():
    attention_output = attention(normalized_hidden)

print("Input:", normalized_hidden.shape)
print("Output:", attention_output.shape)

Input: torch.Size([8, 256, 384])
Output: torch.Size([8, 256, 384])


In [207]:
attention.eval()

original = normalized_hidden.detach()
modified = original.clone()

cutoff = original.shape[1] // 2

modified[:, cutoff:, :] = torch.randn_like(
    modified[:, cutoff:, :]
)

with torch.no_grad():
    original_output = attention(original)
    modified_output = attention(modified)

assert torch.allclose(
    original_output[:, :cutoff, :],
    modified_output[:, :cutoff, :],
    atol=1e-6,
    rtol=1e-5,
)

print("PASS: changing future inputs does not change earlier outputs.")

PASS: changing future inputs does not change earlier outputs.
